# 03 — Real public dataset preprocessing

**Purpose.** Preprocessing of the public healthcare tables: impossible zeros as missing, duplicate removal, train-only imputation and scaling.

**Author:** Guy M. Kaptue T. · **Execution:** independent, seeded (`SEED = 0`), demonstration scale (CPU, about a minute). Outputs: `reports/figures/03_real_dataset_preprocessing_*.png|pdf` and `reports/notebooks/03_real_dataset_preprocessing.json`.

> **Data statement.** SYN-IDS-v1 is completely synthetic despite its name (scikit-learn `make_classification`); it is not real intrusion data and not patient data. The public healthcare datasets are de-identified research tables; no real hospital data is used, all federated clients are simulated locally in one process, and raw source files are not redistributed.

In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
def _root():
    p = Path(os.environ.get("FL_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "pyproject.toml").exists() and (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set FL_ROOT")
ROOT = _root(); sys.path.insert(0, str(ROOT / "src")); os.environ["FL_ROOT"] = str(ROOT)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
SEED = 0; np.random.seed(SEED)
NB_ID = "03_real_dataset_preprocessing"
FIG = ROOT / "reports" / "figures"; OUT = ROOT / "reports" / "notebooks"; FIG.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
PAL = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]   # Okabe-Ito, colour-blind safe
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": matplotlib.cycler(color=PAL), "axes.grid": True, "grid.alpha": .25, "legend.frameon": False})
SUMMARY = {}
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{NB_ID}_{name}.{ext}", bbox_inches="tight")
    plt.show()
def save_summary():
    (OUT / f"{NB_ID}.json").write_text(json.dumps(SUMMARY, indent=2, default=float)); print("summary ->", OUT / f"{NB_ID}.json")

## 1. Missing-value codes and duplicates

In [2]:
from federated_learning.preprocessing.real_datasets import _raw
Xp, yp, _ = _raw("pima"); Xh, yh, _ = _raw("heart")
miss = pd.Series(np.isnan(Xp).mean(0), index=["pregnancies", "glucose", "blood_pressure", "skin", "insulin", "bmi", "pedigree", "age"])
display(miss.round(3).to_frame("missing fraction (zeros recoded)"))
print("Cleveland rows after duplicate removal:", len(yh), "(303 raw)"); SUMMARY.update(pima_missing=miss.to_dict(), cleveland_rows=int(len(yh)))

,missing fraction (zeros recoded)
pregnancies,0.000
glucose,0.007
blood_pressure,0.046
skin,0.296
insulin,0.487
bmi,0.014
pedigree,0.000
age,0.000


Cleveland rows after duplicate removal: 302 (303 raw)


## 2. Train-only imputation and scaling (no leakage)

In [3]:
from sklearn.model_selection import train_test_split
from federated_learning.feature_engineering import preprocess_split
a, b, c, d = train_test_split(Xp, yp, test_size=.3, stratify=yp, random_state=SEED)
A, B = preprocess_split(a, b)
print("train mean ~0:", float(abs(A.mean(0)).max()) < 1e-5, "| no NaN left:", bool(np.isfinite(A).all() and np.isfinite(B).all()))

train mean ~0: True | no NaN left: True


## 3. Figure

In [4]:
fig, ax = plt.subplots(figsize=(4.5, 2.6)); miss.plot.bar(ax=ax, color=PAL[1]); ax.set_ylabel("missing fraction"); ax.set_title("Pima: zeros treated as missing")
save_fig(fig, "pima_missing"); save_summary()

summary -> /Users/guykaptue/Downloads/federated_learning_security/reports/notebooks/03_real_dataset_preprocessing.json


## Limitations
Demonstration scale (few rounds, one seed unless stated): numbers here illustrate the workflow and are not the study's reported results, which come from the full experiment pipeline in `experiments/results/` with multiple seeds and confidence intervals.